In [ ]:
!pip install -q qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from collections import Counter

from skimage.metrics import (
    mean_squared_error as mse,
    peak_signal_noise_ratio as psnr,
    structural_similarity as ssim
)

from qiskit import QuantumCircuit

In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# Same test image used in conventional FRQI
original = X_test[0]
label = y_test[0]

# Resize to 16×16
image = cv2.resize(
    original,
    (16, 16),
    interpolation=cv2.INTER_AREA
)

print("Image shape:", image.shape)
print("Digit:", label)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Image shape: (16, 16)
Digit: 7


In [ ]:
def adaptive_error_partition(
    image,
    min_size=2,
    max_size=8,
    mse_threshold=500
):

    image_float = image.astype(
        np.float32
    )

    adaptive_patches = []

    def process_region(
        x,
        y,
        size
    ):

        patch = image_float[
            y:y+size,
            x:x+size
        ]

        mean_intensity = np.mean(
            patch
        )

        error = np.mean(
            (
                patch -
                mean_intensity
            ) ** 2
        )

        if (
            error > mse_threshold
            and size > min_size
        ):

            new_size = size // 2

            process_region(
                x,
                y,
                new_size
            )

            process_region(
                x + new_size,
                y,
                new_size
            )

            process_region(
                x,
                y + new_size,
                new_size
            )

            process_region(
                x + new_size,
                y + new_size,
                new_size
            )

        else:

            adaptive_patches.append({

                "x": x,
                "y": y,
                "size": size,
                "mse": error
            })

    for y in range(
        0,
        16,
        max_size
    ):

        for x in range(
            0,
            16,
            max_size
        ):

            process_region(
                x,
                y,
                max_size
            )

    return adaptive_patches

In [ ]:
image_16 = cv2.resize(
    X_test[0],
    (16, 16),
    interpolation=cv2.INTER_AREA
)

adaptive_patches = adaptive_error_partition(
    image_16,
    min_size=2,
    max_size=8,
    mse_threshold=500
)

print("Adaptive Error-Based Patching")
print("-----------------------------")

print(
    "Total patches:",
    len(adaptive_patches)
)

patch_distribution = Counter(
    patch["size"]
    for patch in adaptive_patches
)

for size in [8, 4, 2]:

    print(
        f"{size}×{size} patches:",
        patch_distribution.get(size, 0)
    )

Adaptive Error-Based Patching
-----------------------------
Total patches: 31
8×8 patches: 0
4×4 patches: 11
2×2 patches: 20


In [ ]:
multi_value_patches = []

for patch in adaptive_patches:

    x = patch["x"]
    y = patch["y"]
    size = patch["size"]

    patch_image = image_16[
        y:y+size,
        x:x+size
    ]

    if size == 8:

        values = np.array([
            np.mean(patch_image)
        ])

    elif size == 4:

        values = np.array([

            np.mean(
                patch_image[0:2, 0:2]
            ),

            np.mean(
                patch_image[0:2, 2:4]
            ),

            np.mean(
                patch_image[2:4, 0:2]
            ),

            np.mean(
                patch_image[2:4, 2:4]
            )
        ])

    elif size == 2:

        values = patch_image.flatten()

    multi_value_patches.append({

        "x": x,
        "y": y,
        "size": size,
        "values": values,
        "num_values": len(values)
    })

total_values = sum(
    patch["num_values"]
    for patch in multi_value_patches
)

print("Multi-Value Adaptive Representation")
print("-----------------------------------")

print(
    "Total patches:",
    len(multi_value_patches)
)

print(
    "Total intensity values:",
    total_values
)

for size in [8, 4, 2]:

    size_patches = [
        p for p in multi_value_patches
        if p["size"] == size
    ]

    print(
        f"{size}×{size}:",
        len(size_patches),
        "patches"
    )

    print(
        "Values:",
        sum(
            p["num_values"]
            for p in size_patches
        )
    )

Multi-Value Adaptive Representation
-----------------------------------
Total patches: 31
Total intensity values: 124
8×8: 0 patches
Values: 0
4×4: 11 patches
Values: 44
2×2: 20 patches
Values: 80


In [ ]:
intensity_values = []

patch_metadata = []

value_index = 0

for patch in multi_value_patches:

    x = patch["x"]
    y = patch["y"]
    size = patch["size"]

    values = patch["values"]

    for local_index, value in enumerate(values):

        intensity_values.append(
            float(value)
        )

        patch_metadata.append({

            "index": value_index,

            "x": x,

            "y": y,

            "size": size,

            "local_index": local_index,

            "intensity": float(value)
        })

        value_index += 1

intensity_values = np.array(
    intensity_values,
    dtype=np.float64
)

print("Indexed Multi-Value Representation")
print("-----------------------------------")

print(
    "Total intensity values:",
    len(intensity_values)
)

print(
    "Minimum intensity:",
    intensity_values.min()
)

print(
    "Maximum intensity:",
    intensity_values.max()
)

print(
    "First 10 intensity values:",
    np.round(
        intensity_values[:10],
        2
    )
)

Indexed Multi-Value Representation
-----------------------------------
Total intensity values: 124
Minimum intensity: 0.0
Maximum intensity: 252.0
First 10 intensity values: [ 0.   0.   0.   0.   0.   0.   0.   0.   0.  30.5]


In [ ]:
theta_values = (
    intensity_values / 255.0
) * (np.pi / 2)

print("FRQI Angle Encoding")
print("-------------------")

print(
    "Total angles:",
    len(theta_values)
)

print(
    "Minimum angle:",
    theta_values.min()
)

print(
    "Maximum angle:",
    theta_values.max()
)

print(
    "First 10 angles:",
    np.round(
        theta_values[:10],
        4
    )
)

FRQI Angle Encoding
-------------------
Total angles: 124
Minimum angle: 0.0
Maximum angle: 1.5523163700090743
First 10 angles: [0.     0.     0.     0.     0.     0.     0.     0.     0.     0.1879]


In [ ]:
num_values = len(
    intensity_values
)

position_qubits_count = int(
    np.ceil(
        np.log2(num_values)
    )
)

intensity_qubit = 0

position_qubits = list(
    range(
        1,
        position_qubits_count + 1
    )
)

total_qubits = (
    position_qubits_count + 1
)

print("Multi-Value Adaptive-FRQI Resources")
print("------------------------------------")

print(
    "Intensity values:",
    num_values
)

print(
    "Position qubits:",
    position_qubits_count
)

print(
    "Intensity qubits:",
    1
)

print(
    "Total qubits:",
    total_qubits
)

print(
    "Available position states:",
    2 ** position_qubits_count
)

print(
    "Unused position states:",
    (2 ** position_qubits_count) - num_values
)

Multi-Value Adaptive-FRQI Resources
------------------------------------
Intensity values: 124
Position qubits: 7
Intensity qubits: 1
Total qubits: 8
Available position states: 128
Unused position states: 4


In [ ]:
adaptive_qc = QuantumCircuit(
    total_qubits
)

for q in position_qubits:

    adaptive_qc.h(q)

for index, angle in enumerate(
    theta_values
):

    binary = format(
        index,
        f"0{position_qubits_count}b"
    )

    binary = binary[::-1]

    for q, bit in zip(
        position_qubits,
        binary
    ):

        if bit == "0":

            adaptive_qc.x(q)

    adaptive_qc.mcry(
        2 * angle,
        position_qubits,
        intensity_qubit
    )

    for q, bit in zip(
        position_qubits,
        binary
    ):

        if bit == "0":

            adaptive_qc.x(q)

print("Multi-Value Adaptive-FRQI Circuit")
print("----------------------------------")

print(
    "Total qubits:",
    adaptive_qc.num_qubits
)

print(
    "Total gates:",
    adaptive_qc.size()
)

print(
    "Circuit depth:",
    adaptive_qc.depth()
)

Multi-Value Adaptive-FRQI Circuit
----------------------------------
Total qubits: 8
Total gates: 25943
Circuit depth: 21577


In [ ]:
print(adaptive_qc.draw(
    "text",
    fold=120
))

     ┌───┐       ┌───┐   ┌─────┐┌───┐┌───┐┌───┐┌─────┐┌───┐┌───┐ ┌───┐ ┌───┐                                          »
q_0: ┤ H ├───────┤ X ├───┤ Tdg ├┤ X ├┤ T ├┤ X ├┤ Tdg ├┤ X ├┤ T ├─┤ H ├─┤ H ├──────────────────────────────────────────»
     ├───┤┌───┐  └─┬─┘   └─────┘└─┬─┘└───┘└─┬─┘└─────┘└─┬─┘└───┘ └───┘ └───┘                                          »
q_1: ┤ H ├┤ X ├────┼──────────────┼─────────┼───────────┼─────────────────────────────────────────────────────────────»
     ├───┤├───┤    │              │         │           │                                                             »
q_2: ┤ H ├┤ X ├────┼──────────────┼─────────┼───────────┼──────────────────────────────────────────────────────────■──»
     ├───┤├───┤    │              │         │           │                                                          │  »
q_3: ┤ H ├┤ X ├────┼──────────────┼─────────┼───────────┼───────────────────────────────■──────────────────────────┼──»
     ├───┤├───┤    │              │     

In [ ]:
from qiskit.quantum_info import Statevector

statevector = Statevector.from_instruction(
    adaptive_qc
)

print("Statevector simulation completed.")

print(
    "Number of amplitudes:",
    len(statevector.data)
)

Statevector simulation completed.
Number of amplitudes: 256


In [ ]:
reconstructed_values = []

num_positions = 2 ** position_qubits_count

for index in range(num_values):

    state_index = (
        (index << 1) + 1
    )

    probability = (
        abs(
            statevector.data[state_index]
        ) ** 2
    )

    theta = np.arcsin(
        np.sqrt(
            probability * num_positions
        )
    )

    intensity = (
        theta / (np.pi / 2)
    ) * 255.0

    reconstructed_values.append(
        intensity
    )

reconstructed_values = np.array(
    reconstructed_values
)

print("Quantum Intensity Extraction")
print("----------------------------")

print(
    "Extracted values:",
    len(reconstructed_values)
)

print(
    "Minimum intensity:",
    reconstructed_values.min()
)

print(
    "Maximum intensity:",
    reconstructed_values.max()
)

print(
    "First 10 extracted values:",
    np.round(
        reconstructed_values[:10],
        2
    )
)

Quantum Intensity Extraction
----------------------------
Extracted values: 124
Minimum intensity: 4.201115813169798e-12
Maximum intensity: 251.99999999446032
First 10 extracted values: [ 0.   0.   0.   0.   0.   0.   0.   0.   0.  30.5]


In [ ]:
quantum_reconstructed = np.zeros(
    (16, 16),
    dtype=np.float64
)

value_counter = 0

for patch in multi_value_patches:

    x = patch["x"]
    y = patch["y"]
    size = patch["size"]

    if size == 8:

        quantum_reconstructed[
            y:y+8,
            x:x+8
        ] = reconstructed_values[
            value_counter
        ]

        value_counter += 1

    elif size == 4:

        quantum_reconstructed[
            y:y+2,
            x:x+2
        ] = reconstructed_values[
            value_counter
        ]

        quantum_reconstructed[
            y:y+2,
            x+2:x+4
        ] = reconstructed_values[
            value_counter + 1
        ]

        quantum_reconstructed[
            y+2:y+4,
            x:x+2
        ] = reconstructed_values[
            value_counter + 2
        ]

        quantum_reconstructed[
            y+2:y+4,
            x+2:x+4
        ] = reconstructed_values[
            value_counter + 3
        ]

        value_counter += 4

    elif size == 2:

        quantum_reconstructed[
            y:y+2,
            x:x+2
        ] = reconstructed_values[
            value_counter:value_counter + 4
        ].reshape(2, 2)

        value_counter += 4

print("Quantum reconstruction completed.")

print(
    "Values mapped:",
    value_counter
)

Quantum reconstruction completed.
Values mapped: 124


In [ ]:
quantum_mse = mse(
    image_16,
    quantum_reconstructed
)

quantum_psnr = psnr(
    image_16,
    quantum_reconstructed,
    data_range=255
)

quantum_ssim = ssim(
    image_16,
    quantum_reconstructed,
    data_range=255
)

print("Multi-Value Adaptive-FRQI Reconstruction")
print("-----------------------------------------")

print(
    "MSE  :",
    quantum_mse
)

print(
    "PSNR :",
    quantum_psnr,
    "dB"
)

print(
    "SSIM :",
    quantum_ssim
)

Multi-Value Adaptive-FRQI Reconstruction
-----------------------------------------
MSE  : 46.3935546875
PSNR : 31.46622711306207 dB
SSIM : 0.9895736811173202


In [ ]:
multi_value_frqi_results = []

num_images = 100

for idx in range(num_images):

    image_16 = cv2.resize(
        X_test[idx],
        (16, 16),
        interpolation=cv2.INTER_AREA
    )

    adaptive_patches = adaptive_error_partition(
        image_16,
        min_size=2,
        max_size=8,
        mse_threshold=500
    )

    multi_value_patches = []

    for patch in adaptive_patches:

        x = patch["x"]
        y = patch["y"]
        size = patch["size"]

        patch_image = image_16[
            y:y+size,
            x:x+size
        ]

        if size == 8:

            values = np.array([
                np.mean(patch_image)
            ])

        elif size == 4:

            values = np.array([

                np.mean(
                    patch_image[0:2, 0:2]
                ),

                np.mean(
                    patch_image[0:2, 2:4]
                ),

                np.mean(
                    patch_image[2:4, 0:2]
                ),

                np.mean(
                    patch_image[2:4, 2:4]
                )
            ])

        elif size == 2:

            values = patch_image.flatten()

        multi_value_patches.append({

            "x": x,
            "y": y,
            "size": size,
            "values": values
        })

    intensity_values_temp = []

    for patch in multi_value_patches:

        for value in patch["values"]:

            intensity_values_temp.append(
                float(value)
            )

    intensity_values_temp = np.array(
        intensity_values_temp,
        dtype=np.float64
    )

    num_values = len(
        intensity_values_temp
    )

    theta_values_temp = (
        intensity_values_temp / 255.0
    ) * (np.pi / 2)

    position_qubits_count_temp = int(
        np.ceil(
            np.log2(num_values)
        )
    )

    intensity_qubit_temp = 0

    position_qubits_temp = list(
        range(
            1,
            position_qubits_count_temp + 1
        )
    )

    total_qubits_temp = (
        position_qubits_count_temp + 1
    )

    adaptive_qc_temp = QuantumCircuit(
        total_qubits_temp
    )

    for q in position_qubits_temp:

        adaptive_qc_temp.h(q)

    for value_index, angle in enumerate(
        theta_values_temp
    ):

        binary = format(
            value_index,
            f"0{position_qubits_count_temp}b"
        )

        binary = binary[::-1]

        for q, bit in zip(
            position_qubits_temp,
            binary
        ):

            if bit == "0":

                adaptive_qc_temp.x(q)

        adaptive_qc_temp.mcry(
            2 * angle,
            position_qubits_temp,
            intensity_qubit_temp
        )

        for q, bit in zip(
            position_qubits_temp,
            binary
        ):

            if bit == "0":

                adaptive_qc_temp.x(q)

    statevector_temp = Statevector.from_instruction(
        adaptive_qc_temp
    )

    reconstructed_values_temp = []

    num_positions_temp = (
        2 ** position_qubits_count_temp
    )

    for value_index in range(
        num_values
    ):

        state_index = (
            (value_index << 1) + 1
        )

        probability = (
            abs(
                statevector_temp.data[
                    state_index
                ]
            ) ** 2
        )

        theta = np.arcsin(
            np.sqrt(
                probability *
                num_positions_temp
            )
        )

        intensity = (
            theta / (np.pi / 2)
        ) * 255.0

        reconstructed_values_temp.append(
            intensity
        )

    reconstructed_values_temp = np.array(
        reconstructed_values_temp
    )

    quantum_reconstructed_temp = np.zeros(
        (16, 16),
        dtype=np.float64
    )

    value_counter = 0

    for patch in multi_value_patches:

        x = patch["x"]
        y = patch["y"]
        size = patch["size"]

        if size == 8:

            quantum_reconstructed_temp[
                y:y+8,
                x:x+8
            ] = reconstructed_values_temp[
                value_counter
            ]

            value_counter += 1

        elif size == 4:

            quantum_reconstructed_temp[
                y:y+2,
                x:x+2
            ] = reconstructed_values_temp[
                value_counter
            ]

            quantum_reconstructed_temp[
                y:y+2,
                x+2:x+4
            ] = reconstructed_values_temp[
                value_counter + 1
            ]

            quantum_reconstructed_temp[
                y+2:y+4,
                x:x+2
            ] = reconstructed_values_temp[
                value_counter + 2
            ]

            quantum_reconstructed_temp[
                y+2:y+4,
                x+2:x+4
            ] = reconstructed_values_temp[
                value_counter + 3
            ]

            value_counter += 4

        elif size == 2:

            quantum_reconstructed_temp[
                y:y+2,
                x:x+2
            ] = reconstructed_values_temp[
                value_counter:value_counter + 4
            ].reshape(2, 2)

            value_counter += 4

    mse_value = mse(
        image_16,
        quantum_reconstructed_temp
    )

    psnr_value = psnr(
        image_16,
        quantum_reconstructed_temp,
        data_range=255
    )

    ssim_value = ssim(
        image_16,
        quantum_reconstructed_temp,
        data_range=255
    )

    multi_value_frqi_results.append({

        "Image": idx,

        "Adaptive Patches": len(
            multi_value_patches
        ),

        "Intensity Values": num_values,

        "Qubits": adaptive_qc_temp.num_qubits,

        "Gates": adaptive_qc_temp.size(),

        "Depth": adaptive_qc_temp.depth(),

        "MSE": mse_value,

        "PSNR": psnr_value,

        "SSIM": ssim_value
    })

    print(
        f"Image {idx+1}/{num_images} completed",
        end="\r"
    )

multi_value_frqi_results = pd.DataFrame(
    multi_value_frqi_results
)

print(
    "\n\n100-image Multi-Value Adaptive-FRQI evaluation completed."
)

display(
    multi_value_frqi_results.head()
)

Image 100/100 completed

100-image Multi-Value Adaptive-FRQI evaluation completed.


,Image,Adaptive Patches,Intensity Values,Qubits,Gates,Depth,MSE,PSNR,SSIM
0,0,31,124,8,25943,21577,4.639355e+01,31.466227,0.989574
1,1,46,184,9,52416,36259,3.311934e-18,222.929987,1.000000
2,2,28,109,8,22847,18967,1.391052e+00,46.697369,0.997488
3,3,46,184,9,52416,36259,1.120698e-17,217.635919,1.000000
4,4,37,148,9,42208,29163,8.173828e-01,49.006549,0.999887


In [ ]:
display(
    multi_value_frqi_results.head()
)

,Image,Adaptive Patches,Intensity Values,Qubits,Gates,Depth,MSE,PSNR,SSIM
0,0,31,124,8,25943,21577,4.639355e+01,31.466227,0.989574
1,1,46,184,9,52416,36259,3.311934e-18,222.929987,1.000000
2,2,28,109,8,22847,18967,1.391052e+00,46.697369,0.997488
3,3,46,184,9,52416,36259,1.120698e-17,217.635919,1.000000
4,4,37,148,9,42208,29163,8.173828e-01,49.006549,0.999887


In [ ]:
finite_psnr = multi_value_frqi_results[
    np.isfinite(
        multi_value_frqi_results["PSNR"]
    )
]["PSNR"]

exact_reconstructions = (
    multi_value_frqi_results["MSE"] < 1e-10
).sum()

summary = pd.DataFrame({

    "Metric": [
        "Average Adaptive Patches",
        "Average Intensity Values",
        "Average Qubits",
        "Average Gates",
        "Average Depth",
        "Average MSE",
        "Average Finite PSNR",
        "Average SSIM",
        "Exact Reconstructions"
    ],

    "Value": [
        multi_value_frqi_results[
            "Adaptive Patches"
        ].mean(),

        multi_value_frqi_results[
            "Intensity Values"
        ].mean(),

        multi_value_frqi_results[
            "Qubits"
        ].mean(),

        multi_value_frqi_results[
            "Gates"
        ].mean(),

        multi_value_frqi_results[
            "Depth"
        ].mean(),

        multi_value_frqi_results[
            "MSE"
        ].mean(),

        finite_psnr.mean(),

        multi_value_frqi_results[
            "SSIM"
        ].mean(),

        exact_reconstructions
    ]
})

summary["Value"] = summary[
    "Value"
].round(3)

display(summary)

,Metric,Value
0,Average Adaptive Patches,38.320
1,Average Intensity Values,152.890
2,Average Qubits,8.830
3,Average Gates,42142.430
4,Average Depth,29688.000
5,Average MSE,11.700
6,Average Finite PSNR,89.154
7,Average SSIM,0.996
8,Exact Reconstructions,25.000


In [ ]:
print("Quantum Resource Range")
print("---------------------")

print(
    "Qubits:",
    multi_value_frqi_results["Qubits"].min(),
    "to",
    multi_value_frqi_results["Qubits"].max()
)

print(
    "Gates:",
    multi_value_frqi_results["Gates"].min(),
    "to",
    multi_value_frqi_results["Gates"].max()
)

print(
    "Depth:",
    multi_value_frqi_results["Depth"].min(),
    "to",
    multi_value_frqi_results["Depth"].max()
)

print(
    "Intensity Values:",
    multi_value_frqi_results[
        "Intensity Values"
    ].min(),
    "to",
    multi_value_frqi_results[
        "Intensity Values"
    ].max()
)

Quantum Resource Range
---------------------
Qubits: 8 to 9
Gates: 20347 to 59208
Depth: 16879 to 40979
Intensity Values: 97 to 208


In [ ]:
multi_value_frqi_results.to_csv(
    "multi_value_adaptive_frqi_100_images.csv",
    index=False
)

from google.colab import files

files.download(
    "multi_value_adaptive_frqi_100_images.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FINAL COMPARISON

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving adaptive_frqi_100_images (2).csv to adaptive_frqi_100_images (2).csv
Saving conventional_frqi_100_images.csv to conventional_frqi_100_images (1).csv


In [ ]:
import pandas as pd

conventional_results = pd.read_csv(
    "conventional_frqi_100_images.csv"
)

adaptive_frqi_results = pd.read_csv(
    "adaptive_frqi_100_images (2).csv"
)

print("Conventional FRQI")
display(
    conventional_results.head()
)

print("Adaptive-FRQI")
display(
    adaptive_frqi_results.head()
)

Conventional FRQI


,Image,Digit,Pixels,Qubits,Gates,Depth
0,0,7,256,9,72712,50435
1,1,2,256,9,72712,50435
2,2,1,256,9,72712,50435
3,3,0,256,9,72712,50435
4,4,4,256,9,72712,50435


Adaptive-FRQI


,Image,Digit,Adaptive Patches,Qubits,Gates,Depth
0,0,7,31,6,3141,1871
1,1,2,46,7,6208,3507
2,2,1,28,6,2845,1691
3,3,0,46,7,6208,3507
4,4,4,37,7,5006,2823


In [ ]:
print("Conventional columns:")
print(
    conventional_results.columns.tolist()
)

print("\nAdaptive columns:")
print(
    adaptive_frqi_results.columns.tolist()
)

print("\nMulti-Value Adaptive columns:")
print(
    multi_value_frqi_results.columns.tolist()
)

Conventional columns:
['Image', 'Digit', 'Pixels', 'Qubits', 'Gates', 'Depth']

Adaptive columns:
['Image', 'Digit', 'Adaptive Patches', 'Qubits', 'Gates', 'Depth']

Multi-Value Adaptive columns:
['Image', 'Adaptive Patches', 'Intensity Values', 'Qubits', 'Gates', 'Depth', 'MSE', 'PSNR', 'SSIM']


In [ ]:
comparison = pd.DataFrame({

    "Method": [
        "Conventional FRQI",
        "Adaptive-FRQI",
        "Multi-Value Adaptive-FRQI"
    ],

    "Average Elements": [
        conventional_results["Pixels"].mean(),
        adaptive_frqi_results["Adaptive Patches"].mean(),
        multi_value_frqi_results["Intensity Values"].mean()
    ],

    "Average Qubits": [
        conventional_results["Qubits"].mean(),
        adaptive_frqi_results["Qubits"].mean(),
        multi_value_frqi_results["Qubits"].mean()
    ],

    "Average Gates": [
        conventional_results["Gates"].mean(),
        adaptive_frqi_results["Gates"].mean(),
        multi_value_frqi_results["Gates"].mean()
    ],

    "Average Depth": [
        conventional_results["Depth"].mean(),
        adaptive_frqi_results["Depth"].mean(),
        multi_value_frqi_results["Depth"].mean()
    ]
})

comparison = comparison.round(2)

print("===Resource Comparison===")
display(comparison)

===Resource Comparison===


,Method,Average Elements,Average Qubits,Average Gates,Average Depth
0,Conventional FRQI,256.00,9.00,72712.00,50435.00
1,Adaptive-FRQI,38.32,6.83,5015.23,2845.72
2,Multi-Value Adaptive-FRQI,152.89,8.83,42142.43,29688.00


In [ ]:
reconstruction_results = []

num_images = 100

for idx in range(num_images):

    image_16 = cv2.resize(
        X_test[idx],
        (16, 16),
        interpolation=cv2.INTER_AREA
    )

    conventional_reconstructed = (
        image_16.astype(np.float64)
    )

    conventional_mse = mse(
        image_16,
        conventional_reconstructed
    )

    conventional_psnr = psnr(
        image_16,
        conventional_reconstructed,
        data_range=255
    )

    conventional_ssim = ssim(
        image_16,
        conventional_reconstructed,
        data_range=255
    )

    adaptive_patches = adaptive_error_partition(
        image_16,
        min_size=2,
        max_size=8,
        mse_threshold=500
    )

    adaptive_reconstructed = np.zeros(
        (16, 16),
        dtype=np.float64
    )

    for patch in adaptive_patches:

        x = patch["x"]
        y = patch["y"]
        size = patch["size"]

        patch_image = image_16[
            y:y+size,
            x:x+size
        ]

        intensity = np.mean(
            patch_image
        )

        adaptive_reconstructed[
            y:y+size,
            x:x+size
        ] = intensity

    adaptive_mse = mse(
        image_16,
        adaptive_reconstructed
    )

    adaptive_psnr = psnr(
        image_16,
        adaptive_reconstructed,
        data_range=255
    )

    adaptive_ssim = ssim(
        image_16,
        adaptive_reconstructed,
        data_range=255
    )

    multi_value_row = (
        multi_value_frqi_results[
            multi_value_frqi_results["Image"] == idx
        ]
    )

    reconstruction_results.append({

        "Image": idx,

        "Conventional MSE":
            conventional_mse,

        "Conventional PSNR":
            conventional_psnr,

        "Conventional SSIM":
            conventional_ssim,

        "Adaptive MSE":
            adaptive_mse,

        "Adaptive PSNR":
            adaptive_psnr,

        "Adaptive SSIM":
            adaptive_ssim,

        "Multi-Value MSE":
            multi_value_row["MSE"].iloc[0],

        "Multi-Value PSNR":
            multi_value_row["PSNR"].iloc[0],

        "Multi-Value SSIM":
            multi_value_row["SSIM"].iloc[0]
    })

    print(
        f"Image {idx+1}/{num_images} completed",
        end="\r"
    )

reconstruction_results = pd.DataFrame(
    reconstruction_results
)

print(
    "\n\n100-image reconstruction comparison completed."
)

display(
    reconstruction_results.head()
)

/usr/local/lib/python3.12/dist-packages/skimage/metrics/simple_metrics.py:168: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


Image 100/100 completed

100-image reconstruction comparison completed.


,Image,Conventional MSE,Conventional PSNR,Conventional SSIM,Adaptive MSE,Adaptive PSNR,Adaptive SSIM,Multi-Value MSE,Multi-Value PSNR,Multi-Value SSIM
0,0,0.0,inf,1.0,785.186279,19.181077,0.829278,4.639355e+01,31.466227,0.989574
1,1,0.0,inf,1.0,1746.401367,15.709363,0.750663,3.311934e-18,222.929987,1.000000
2,2,0.0,inf,1.0,560.501160,20.645038,0.727322,1.391052e+00,46.697369,0.997488
3,3,0.0,inf,1.0,1463.101562,16.478059,0.845667,1.120698e-17,217.635919,1.000000
4,4,0.0,inf,1.0,1386.906006,16.710333,0.691084,8.173828e-01,49.006549,0.999887


In [ ]:
print("Average Reconstruction Results")
print("------------------------------")

reconstruction_summary = pd.DataFrame({

    "Method": [
        "Conventional FRQI",
        "Adaptive-FRQI",
        "Multi-Value Adaptive-FRQI"
    ],

    "Average MSE": [
        reconstruction_results[
            "Conventional MSE"
        ].mean(),

        reconstruction_results[
            "Adaptive MSE"
        ].mean(),

        reconstruction_results[
            "Multi-Value MSE"
        ].mean()
    ],

    "Average PSNR": [
        reconstruction_results[
            "Conventional PSNR"
        ].replace(
            [np.inf, -np.inf],
            np.nan
        ).mean(),

        reconstruction_results[
            "Adaptive PSNR"
        ].replace(
            [np.inf, -np.inf],
            np.nan
        ).mean(),

        reconstruction_results[
            "Multi-Value PSNR"
        ].replace(
            [np.inf, -np.inf],
            np.nan
        ).mean()
    ],

    "Average SSIM": [
        reconstruction_results[
            "Conventional SSIM"
        ].mean(),

        reconstruction_results[
            "Adaptive SSIM"
        ].mean(),

        reconstruction_results[
            "Multi-Value SSIM"
        ].mean()
    ]
})

reconstruction_summary = (
    reconstruction_summary.round(4)
)

display(
    reconstruction_summary
)

Average Reconstruction Results
------------------------------


,Method,Average MSE,Average PSNR,Average SSIM
0,Conventional FRQI,0.000,NaN,1.0000
1,Adaptive-FRQI,1217.012,17.4759,0.7684
2,Multi-Value Adaptive-FRQI,11.700,89.1541,0.9962


In [ ]:
psnr_values = reconstruction_results[
    "Multi-Value PSNR"
]

finite_psnr = psnr_values[
    np.isfinite(psnr_values)
]

print("Multi-Value Adaptive-FRQI PSNR Analysis")
print("----------------------------------------")

print(
    "Finite PSNR images:",
    len(finite_psnr)
)

print(
    "Exact reconstructions:",
    len(psnr_values) - len(finite_psnr)
)

print(
    "Mean finite PSNR:",
    finite_psnr.mean()
)

print(
    "Median finite PSNR:",
    finite_psnr.median()
)

print(
    "Minimum finite PSNR:",
    finite_psnr.min()
)

print(
    "Maximum finite PSNR:",
    finite_psnr.max()
)

Multi-Value Adaptive-FRQI PSNR Analysis
----------------------------------------
Finite PSNR images: 100
Exact reconstructions: 0
Mean finite PSNR: 89.15413238809352
Median finite PSNR: 41.4879487069509
Minimum finite PSNR: 27.97363160831094
Maximum finite PSNR: 252.08530846402817


In [ ]:
final_results = pd.DataFrame({

    "Method": [
        "Conventional FRQI",
        "Adaptive-FRQI",
        "Multi-Value Adaptive-FRQI"
    ],

    "Average Elements": [
        comparison.loc[0, "Average Elements"],
        comparison.loc[1, "Average Elements"],
        comparison.loc[2, "Average Elements"]
    ],

    "Average Qubits": [
        comparison.loc[0, "Average Qubits"],
        comparison.loc[1, "Average Qubits"],
        comparison.loc[2, "Average Qubits"]
    ],

    "Average Gates": [
        comparison.loc[0, "Average Gates"],
        comparison.loc[1, "Average Gates"],
        comparison.loc[2, "Average Gates"]
    ],

    "Average Depth": [
        comparison.loc[0, "Average Depth"],
        comparison.loc[1, "Average Depth"],
        comparison.loc[2, "Average Depth"]
    ],

    "Average MSE": [
        reconstruction_summary.loc[0, "Average MSE"],
        reconstruction_summary.loc[1, "Average MSE"],
        reconstruction_summary.loc[2, "Average MSE"]
    ],

    "PSNR": [
        "∞",
        reconstruction_summary.loc[1, "Average PSNR"],
        41.4879
    ],

    "Average SSIM": [
        reconstruction_summary.loc[0, "Average SSIM"],
        reconstruction_summary.loc[1, "Average SSIM"],
        reconstruction_summary.loc[2, "Average SSIM"]
    ]
})

final_results = final_results.round(4)

display(final_results)

,Method,Average Elements,Average Qubits,Average Gates,Average Depth,Average MSE,PSNR,Average SSIM
0,Conventional FRQI,256.00,9.00,72712.00,50435.00,0.000,∞,1.0000
1,Adaptive-FRQI,38.32,6.83,5015.23,2845.72,1217.012,17.4759,0.7684
2,Multi-Value Adaptive-FRQI,152.89,8.83,42142.43,29688.00,11.700,41.4879,0.9962
